In [4]:
pip install requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   -------- ------------------------------- 1/5 [idna]
   -------- ------------------------------- 1/5 [idna]
   ---

In [6]:
!pip install beautifulsoup4 requests 

  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)

   ---------------------------------------- 0/2 [soupsieve]
   ---------------------------------------- 0/2 [soupsieve]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   ---------------------------------------- 2/2 [beautifulsoup4]



In [1]:
import sqlite3
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

In [2]:
BASE = "https://books.toscrape.com/"
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
GBP_TO_INR = 105.50                     # required fixed baseline (project constant)
HEADERS = {"User-Agent": "Mozilla/5.0 (educational scraper)"}

In [3]:
session = requests.Session()
session.headers.update(HEADERS)

def get_soup(url):
    r = session.get(url, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

In [4]:
home = get_soup(BASE)
cat_links = home.select(".side_categories ul li ul li a")
categories = [(a.get_text(strip=True), BASE + a["href"]) for a in cat_links[:5]]
print("Categories:", [c[0] for c in categories])

Categories: ['Travel', 'Mystery', 'Historical Fiction', 'Sequential Art', 'Classics']


In [5]:
rows = []
for cat_name, cat_url in categories:
    page_url = cat_url
    page_num = 0
    while page_url and page_num < 5:            # cap pages per category
        page_num += 1
        soup = get_soup(page_url)
        for card in soup.select("article.product_pod"):
            rows.append({
                "title":             card.h3.a["title"].strip(),
                "price_text":        card.select_one(".price_color").get_text(strip=True),
                "rating_text":       card.select_one(".star-rating")["class"][1],
                "availability_text": card.select_one(".availability").get_text(strip=True),
                "category":          cat_name,
            })
        nxt = soup.select_one("li.next a")
        page_url = cat_url.rsplit("/", 1)[0] + "/" + nxt["href"] if nxt else None
        time.sleep(0.3)                          # polite pause

raw = pd.DataFrame(rows)
print("raw rows scraped:", len(raw))
raw.head()

raw rows scraped: 163


,title,price_text,rating_text,availability_text,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mount Ararat ...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Parks & Treas...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of Long-Term W...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel


In [6]:
def parse_price(t):
    try:
        return float(t.replace("£", "").replace("Â", "").strip())
    except (ValueError, AttributeError):
        return None

raw["price_gbp"] = raw["price_text"].apply(parse_price)
raw["rating"]    = raw["rating_text"].map(RATING_MAP)
raw["in_stock"]  = raw["availability_text"].str.contains("In stock", case=False, na=False)

# Bad-parse handling:
# - price_gbp: median imputation (keeps row, robust to skew)
# - rating:    drop row (fabricating a rating would distort filters/joins)
before = len(raw)
price_median = raw["price_gbp"].median()
n_price_fix  = raw["price_gbp"].isna().sum()
raw["price_gbp"] = raw["price_gbp"].fillna(price_median)
raw = raw.dropna(subset=["rating"])
raw["rating"] = raw["rating"].astype(int)

print(f"price median = {price_median:.2f}; imputed for {n_price_fix} rows")
print(f"dropped {before - len(raw)} rows with unparseable rating")
raw[["title","price_gbp","rating","in_stock","category"]].head()

price median = 35.28; imputed for 0 rows
dropped 0 rows with unparseable rating


,title,price_gbp,rating,in_stock,category
0,It's Only the Himalayas,45.17,2,True,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mount Ararat ...,49.43,4,True,Travel
2,See America: A Celebration of Our National Parks & Treas...,48.87,3,True,Travel
3,Vagabonding: An Uncommon Guide to the Art of Long-Term W...,36.94,2,True,Travel
4,Under the Tuscan Sun,37.33,3,True,Travel


In [7]:
raw["price_inr"] = (raw["price_gbp"] * GBP_TO_INR).round(2)
print("Fixed conversion applied: 1 GBP = 105.50 INR")

books = raw[["title","price_gbp","price_inr","rating","in_stock","category"]].copy()
books["in_stock"] = books["in_stock"].astype(int)          # 0/1 for SQLite
books.head()

Fixed conversion applied: 1 GBP = 105.50 INR


,title,price_gbp,price_inr,rating,in_stock,category
0,It's Only the Himalayas,45.17,4765.44,2,1,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mount Ararat ...,49.43,5214.86,4,1,Travel
2,See America: A Celebration of Our National Parks & Treas...,48.87,5155.78,3,1,Travel
3,Vagabonding: An Uncommon Guide to the Art of Long-Term W...,36.94,3897.17,2,1,Travel
4,Under the Tuscan Sun,37.33,3938.31,3,1,Travel


In [8]:
con = sqlite3.connect("books.db")
cur = con.cursor()
cur.executescript("""
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS categories;

CREATE TABLE categories (
    category_id   INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE NOT NULL
);

CREATE TABLE books (
    book_id     INTEGER PRIMARY KEY,
    title       TEXT NOT NULL,
    price_gbp   REAL NOT NULL,
    price_inr   REAL NOT NULL,
    rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    in_stock    INTEGER NOT NULL CHECK (in_stock IN (0,1)),
    category_id INTEGER NOT NULL REFERENCES categories(category_id)
);
""")
print("schema created")

schema created


In [9]:
cats = sorted(books["category"].unique())
cur.executemany("INSERT INTO categories(category_name) VALUES (?)", [(c,) for c in cats])

cat_id = {name: cid for cid, name in cur.execute(
    "SELECT category_id, category_name FROM categories")}

book_rows = [
    (r.title, float(r.price_gbp), float(r.price_inr),
     int(r.rating), int(r.in_stock), cat_id[r.category])
    for r in books.itertuples(index=False)
]
cur.executemany(
    "INSERT INTO books(title, price_gbp, price_inr, rating, in_stock, category_id) "
    "VALUES (?, ?, ?, ?, ?, ?)", book_rows
)
con.commit()

In [10]:
n_books = cur.execute("SELECT COUNT(*) FROM books").fetchone()[0]
n_cats  = cur.execute("SELECT COUNT(*) FROM categories").fetchone()[0]
print(f"loaded {n_books} books across {n_cats} categories into books.db")
con.close()

loaded 163 books across 5 categories into books.db
